# ML-Based Imputation of Missing Nutrition Data in Humanitarian Surveys
### By Joshua Samuel Dauda
---

> *'Machine Learning-Based Imputation of Missing Household Nutrition Data in Humanitarian Surveys: A Comparative Study Using DHS-Consistent Data from Northeast Nigeria'*

**Methods compared:** Mean Imputation (Baseline) | KNN Imputation | MICE | Random Forest Imputation

**Target variable:** Weight-for-Height Z-score (WHZ) — proxy for acute malnutrition (wasting)

**Dataset:** Simulated DHS-consistent dataset (structure mirrors Nigeria DHS 2018 child nutrition module)

## 0. Environment Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import BayesianRidge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

np.random.seed(42)
print('Environment ready.')

## 1. Data Generation
Simulated a DHS-consistent child nutrition dataset reflecting the demographic and nutritional structure of Northeast Nigeria (Borno, Adamawa, Yobe states).

**Variables:**
- `age_months`, `sex`, `birth_order`, `hh_size`
- `mother_edu`: 0=None, 1=Primary, 2=Secondary, 3=Higher
- `wealth_index`: 1 (poorest) to 5 (richest)
- `water_source`, `diarrhea_2wk`, `breastfed`
- `muac_cm`: Mid-Upper Arm Circumference
- `haz`: Height-for-Age Z-score
- `whz`: **TARGET — Weight-for-Height Z-score**

In [ ]:
N = 1500
age_months   = np.random.randint(6, 60, N)
sex          = np.random.binomial(1, 0.51, N)
birth_order  = np.random.randint(1, 7, N)
hh_size      = np.random.randint(4, 13, N)
mother_edu   = np.random.choice([0,1,2,3], N, p=[0.55,0.25,0.15,0.05])
wealth_index = np.random.choice([1,2,3,4,5], N, p=[0.35,0.28,0.20,0.11,0.06])
water_source = np.random.binomial(1, 0.42, N)
diarrhea_2wk = np.random.binomial(1, 0.28, N)
breastfed    = np.where(age_months < 24, np.random.binomial(1,0.72,N), np.random.binomial(1,0.15,N))

haz = (-1.2 + 0.15*mother_edu + 0.10*wealth_index - 0.30*diarrhea_2wk
       + 0.05*water_source - 0.02*birth_order + np.random.normal(0, 0.9, N))

muac_cm = (13.5 + 0.08*age_months - 0.003*age_months**2/10
           + 0.12*wealth_index + 0.08*mother_edu - 0.25*diarrhea_2wk
           + np.random.normal(0, 0.6, N)).clip(9, 20)

whz = (-0.6 + 0.50*haz + 0.30*(muac_cm-13.5) + 0.12*wealth_index
       + 0.10*mother_edu - 0.20*diarrhea_2wk + 0.05*water_source
       - 0.10*(birth_order>3).astype(int) + np.random.normal(0, 0.7, N))

df_complete = pd.DataFrame({
    'age_months':age_months,'sex':sex,'birth_order':birth_order,'hh_size':hh_size,
    'mother_edu':mother_edu,'wealth_index':wealth_index,'water_source':water_source,
    'diarrhea_2wk':diarrhea_2wk,'breastfed':breastfed,'muac_cm':muac_cm,'haz':haz,'whz':whz
})

print(f'Dataset: {df_complete.shape[0]} rows x {df_complete.shape[1]} columns')
print(f'GAM prevalence (WHZ < -2): {(df_complete["whz"] < -2).mean()*100:.1f}%')
print(f'SAM prevalence (WHZ < -3): {(df_complete["whz"] < -3).mean()*100:.1f}%')

## 2. Introduce Missing Data (Three Mechanisms)

In [ ]:
def introduce_missing(df, target_col, missing_rate=0.30, mechanism='MAR'):
    df_missing = df.copy()
    n = len(df)
    if mechanism == 'MCAR':
        idx = np.random.choice(n, int(n*missing_rate), replace=False)
    elif mechanism == 'MAR':
        prob = (0.5 - 0.08*df['wealth_index']).clip(0.05, 0.70)
        idx = np.where(np.random.binomial(1, prob)==1)[0]
    elif mechanism == 'MNAR':
        prob = np.where(df[target_col]<-2, 0.55, 0.15)
        idx = np.where(np.random.binomial(1, prob)==1)[0]
    df_missing.loc[idx, target_col] = np.nan
    return df_missing, df_missing[target_col].isna().mean()

df_mcar, r1 = introduce_missing(df_complete, 'whz', 0.30, 'MCAR')
df_mar,  r2 = introduce_missing(df_complete, 'whz', 0.30, 'MAR')
df_mnar, r3 = introduce_missing(df_complete, 'whz', 0.30, 'MNAR')

print(f'MCAR: {r1*100:.1f}% | MAR: {r2*100:.1f}% | MNAR: {r3*100:.1f}% missing')

## 3. Visualize Missing Patterns by Wealth Quintile

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (mech, df_m, col) in zip(axes, [('MCAR',df_mcar,'#2196F3'),('MAR',df_mar,'#FF9800'),('MNAR',df_mnar,'#F44336')]):
    mb = df_m.groupby('wealth_index')['whz'].apply(lambda x: x.isna().mean()*100)
    ax.bar(mb.index, mb.values, color=col, alpha=0.8, edgecolor='white')
    ax.set_title(f'{mech}\nMissingness by Wealth Quintile', fontweight='bold')
    ax.set_xlabel('Wealth Quintile'); ax.set_ylabel('% WHZ Missing')
    ax.set_ylim(0, 80); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.suptitle('Figure 1: Missing Data Patterns', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/home/claude/fig1_missing_patterns.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Evaluation Function

In [ ]:
feature_cols = ['age_months','sex','birth_order','hh_size','mother_edu',
                'wealth_index','water_source','diarrhea_2wk','breastfed','muac_cm','haz']
target_col = 'whz'
all_cols = feature_cols + [target_col]

def get_imputers():
    return {
        'Mean Imputation':       SimpleImputer(strategy='mean'),
        'KNN (k=5)':             KNNImputer(n_neighbors=5),
        'MICE (BayesianRidge)':  IterativeImputer(estimator=BayesianRidge(), max_iter=10, random_state=42),
        'RF Imputation':         IterativeImputer(estimator=RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1), max_iter=5, random_state=42)
    }

def evaluate_imputation(df_complete, df_missing, imputers):
    mask = df_missing[target_col].isna()
    true_vals = df_complete.loc[mask, target_col].values
    results = []
    for name, imputer in imputers.items():
        arr = imputer.fit_transform(df_missing[all_cols].copy())
        preds = pd.DataFrame(arr, columns=all_cols).loc[mask.values, target_col].values
        results.append({
            'Method': name,
            'RMSE':  round(np.sqrt(mean_squared_error(true_vals, preds)), 4),
            'MAE':   round(mean_absolute_error(true_vals, preds), 4),
            'R2':    round(r2_score(true_vals, preds), 4),
            'Bias':  round(np.mean(preds - true_vals), 4)
        })
    return pd.DataFrame(results).sort_values('RMSE')

print('Functions defined.')

## 5. Run Comparison — All Three Mechanisms

In [ ]:
datasets = {'MCAR (30%)': df_mcar, 'MAR (30%)': df_mar, 'MNAR (30%)': df_mnar}
all_results = {}

for name, df_m in datasets.items():
    print(f'\n--- {name} ---')
    all_results[name] = evaluate_imputation(df_complete, df_m, get_imputers())
    print(all_results[name].to_string(index=False))

## 6. Figure 2 — RMSE Comparison Bar Chart

In [ ]:
methods = ['Mean Imputation','KNN (k=5)','MICE (BayesianRidge)','RF Imputation']
mechs   = list(all_results.keys())
palette = ['#E53935','#FB8C00','#1E88E5','#43A047']

fig, ax = plt.subplots(figsize=(11, 6))
x = np.arange(len(mechs)); width = 0.18

for i, (m, c) in enumerate(zip(methods, palette)):
    vals = [all_results[mech][all_results[mech]['Method']==m]['RMSE'].values[0] for mech in mechs]
    bars = ax.bar(x+i*width, vals, width, label=m, color=c, alpha=0.88, edgecolor='white')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003, f'{v:.3f}', ha='center', fontsize=8.5, fontweight='bold')

ax.set_xticks(x+width*1.5); ax.set_xticklabels(mechs, fontsize=12)
ax.set_ylabel('RMSE (WHZ units)', fontsize=12)
ax.set_title('Figure 2: RMSE by Imputation Method and Missing Data Mechanism', fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('/home/claude/fig2_rmse_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Figure 3 — Scatter: True vs Imputed WHZ (MAR)

In [ ]:
mask_plot = df_mar[target_col].isna()
true_v    = df_complete.loc[mask_plot, target_col].values

mean_arr = SimpleImputer(strategy='mean').fit_transform(df_mar[all_cols].copy())
mean_p   = pd.DataFrame(mean_arr, columns=all_cols).loc[mask_plot.values, target_col].values

rf_arr = IterativeImputer(estimator=RandomForestRegressor(n_estimators=50,random_state=42,n_jobs=-1),max_iter=5,random_state=42).fit_transform(df_mar[all_cols].copy())
rf_p   = pd.DataFrame(rf_arr, columns=all_cols).loc[mask_plot.values, target_col].values

fig, axes = plt.subplots(1, 2, figsize=(13, 5)); lims = (-4.5, 2.5)
for ax, preds, method, color in zip(axes, [mean_p,rf_p], ['Mean Imputation (Baseline)','Random Forest Imputation'], ['#E53935','#43A047']):
    ax.scatter(true_v, preds, alpha=0.35, s=18, color=color, edgecolors='none')
    ax.plot(lims, lims, 'k--', lw=1.5)
    ax.axhline(-2, color='gray', ls=':', lw=1, alpha=0.7)
    ax.axvline(-2, color='gray', ls=':', lw=1, alpha=0.7)
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel('True WHZ', fontsize=12); ax.set_ylabel('Imputed WHZ', fontsize=12)
    ax.set_title(f'{method}\nR²={r2_score(true_v,preds):.3f} | RMSE={np.sqrt(mean_squared_error(true_v,preds)):.3f}', fontsize=11, fontweight='bold')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.suptitle('Figure 3: True vs Imputed WHZ — MAR Mechanism', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/home/claude/fig3_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Figure 4 — GAM Prevalence Estimation Error

In [ ]:
true_gam = (df_complete['whz'] < -2).mean() * 100
gam_results = {}

for mech_name, df_m in datasets.items():
    gam_results[mech_name] = {}
    for method_name, imputer in get_imputers().items():
        arr = imputer.fit_transform(df_m[all_cols].copy())
        gam_prev = (pd.DataFrame(arr, columns=all_cols)['whz'] < -2).mean() * 100
        gam_results[mech_name][method_name] = gam_prev

fig, ax = plt.subplots(figsize=(11, 6))
x = np.arange(len(methods)); width = 0.22
for i, (mech, color) in enumerate(zip(mechs, ['#1E88E5','#FB8C00','#E53935'])):
    ax.bar(x+i*width, [gam_results[mech][m] for m in methods], width, label=mech, color=color, alpha=0.82, edgecolor='white')
ax.axhline(true_gam, color='black', ls='--', lw=2, label=f'True GAM ({true_gam:.1f}%)')
ax.set_xticks(x+width); ax.set_xticklabels(methods, fontsize=10, rotation=10)
ax.set_ylabel('Estimated GAM Prevalence (%)', fontsize=12)
ax.set_title('Figure 4: GAM Prevalence Estimation by Imputation Method', fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('/home/claude/fig4_gam_prevalence.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Sensitivity Analysis — Varying Missingness Rate

In [ ]:
missing_rates = [0.10, 0.20, 0.30, 0.40, 0.50]
sens = {m: [] for m in methods}

for rate in missing_rates:
    df_temp, _ = introduce_missing(df_complete, 'whz', rate, 'MAR')
    mask = df_temp['whz'].isna()
    tv = df_complete.loc[mask, 'whz'].values
    for mname, imp in get_imputers().items():
        arr = imp.fit_transform(df_temp[all_cols].copy())
        pv = pd.DataFrame(arr, columns=all_cols).loc[mask.values, 'whz'].values
        sens[mname].append(np.sqrt(mean_squared_error(tv, pv)))

fig, ax = plt.subplots(figsize=(10, 5))
for m, ls, col in zip(methods, ['-o','-s','-^','-D'], ['#E53935','#FB8C00','#1E88E5','#43A047']):
    ax.plot([r*100 for r in missing_rates], sens[m], ls, label=m, color=col, lw=2.2, markersize=7)
ax.set_xlabel('Missingness Rate (%)', fontsize=12); ax.set_ylabel('RMSE', fontsize=12)
ax.set_title('Figure 5: RMSE vs Missingness Rate (MAR Mechanism)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('/home/claude/fig5_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Figure 6 — Feature Importance

In [ ]:
rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(df_complete[feature_cols].values, df_complete[target_col].values)

imp_df = pd.DataFrame({'Feature': feature_cols, 'Importance': rf.feature_importances_}).sort_values('Importance')

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(imp_df['Feature'], imp_df['Importance'], color='#1E88E5', alpha=0.85, edgecolor='white')
for bar, v in zip(bars, imp_df['Importance']):
    ax.text(bar.get_width()+0.002, bar.get_y()+bar.get_height()/2, f'{v:.3f}', va='center', fontsize=9)
ax.set_xlabel('Feature Importance', fontsize=11)
ax.set_title('Figure 6: RF Feature Importance for WHZ Prediction', fontsize=12, fontweight='bold')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('/home/claude/fig6_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('All figures generated.')

---
## End of Notebook


> Dauda, J.S. (2025). *Machine Learning-Based Imputation of Missing Household Nutrition Data in Humanitarian Surveys.*